# Glucose Prediction — Model Improvement

## Objective

Improve the baseline Random Forest model for 30-minute-ahead glucose prediction through hyperparameter tuning.

## Baseline Model

Random Forest Regressor

Baseline performance:

- MAE = 17.57 mg/dL
- RMSE = 26.41 mg/dL
- R² = 0.8155

## Improvement Strategy

1. Load the processed ML dataset
2. Recreate the patient-wise chronological split
3. Separate training data into training and validation sets
4. Tune Random Forest hyperparameters
5. Evaluate the tuned model
6. Compare tuned model against the baseline
7. Save the improved model only if it performs better

In [1]:
import os
import joblib
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

print("Libraries imported successfully.")

Libraries imported successfully.


In [ ]:
DATA_PATH = "../data/processed/ml_dataset.csv"

df = pd.read_csv(DATA_PATH)

df["timestamp"] = pd.to_datetime(df["timestamp"])

df = df.sort_values(
    ["Patient", "timestamp"]
).reset_index(drop=True)

print("Dataset loaded successfully.")
print("Shape:", df.shape)

Dataset loaded successfully.
Shape: (65499, 21)


In [3]:
feature_columns = [
    "glucose_value",
    "glucose_lag_1",
    "glucose_lag_2",
    "glucose_lag_3",
    "glucose_lag_6",
    "glucose_change_5min",
    "glucose_change_15min",
    "meal_carbs",
    "bolus_dose",
    "basal_value",
    "temp_basal_value",
    "exercise_duration",
    "exercise_intensity",
    "sleep_duration_hours",
    "sleep_quality",
    "finger_stick_value",
    "hour",
    "day_of_week"
]

target_column = "target_glucose_30min"

print("Number of features:", len(feature_columns))
print("Target:", target_column)

Number of features: 18
Target: target_glucose_30min


In [4]:
train_parts = []
test_parts = []

for patient in df["Patient"].unique():

    patient_df = df[
        df["Patient"] == patient
    ].sort_values("timestamp")

    split_index = int(len(patient_df) * 0.80)

    train_parts.append(
        patient_df.iloc[:split_index]
    )

    test_parts.append(
        patient_df.iloc[split_index:]
    )

train_df = pd.concat(train_parts).reset_index(drop=True)

test_df = pd.concat(test_parts).reset_index(drop=True)

print("Training shape:", train_df.shape)
print("Testing shape:", test_df.shape)

Training shape: (52396, 21)
Testing shape: (13103, 21)


In [5]:
X_train_full = train_df[feature_columns]
y_train_full = train_df[target_column]

X_test = test_df[feature_columns]
y_test = test_df[target_column]

print("X_train:", X_train_full.shape)
print("y_train:", y_train_full.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (52396, 18)
y_train: (52396,)
X_test: (13103, 18)
y_test: (13103,)


In [6]:
validation_parts = []
training_parts = []

for patient in train_df["Patient"].unique():

    patient_df = train_df[
        train_df["Patient"] == patient
    ].sort_values("timestamp")

    split_index = int(len(patient_df) * 0.80)

    training_parts.append(
        patient_df.iloc[:split_index]
    )

    validation_parts.append(
        patient_df.iloc[split_index:]
    )

tuning_train_df = pd.concat(
    training_parts
).reset_index(drop=True)

validation_df = pd.concat(
    validation_parts
).reset_index(drop=True)

X_tune_train = tuning_train_df[feature_columns]
y_tune_train = tuning_train_df[target_column]

X_validation = validation_df[feature_columns]
y_validation = validation_df[target_column]

print("Tuning training:", X_tune_train.shape)
print("Validation:", X_validation.shape)
print("Final test:", X_test.shape)

Tuning training: (41914, 18)
Validation: (10482, 18)
Final test: (13103, 18)


In [7]:
baseline_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

print("Training baseline Random Forest...")

baseline_model.fit(
    X_tune_train,
    y_tune_train
)

baseline_validation_predictions = (
    baseline_model.predict(X_validation)
)

baseline_mae = mean_absolute_error(
    y_validation,
    baseline_validation_predictions
)

baseline_rmse = np.sqrt(
    mean_squared_error(
        y_validation,
        baseline_validation_predictions
    )
)

baseline_r2 = r2_score(
    y_validation,
    baseline_validation_predictions
)

print("\nBaseline validation performance:")
print(f"MAE  : {baseline_mae:.4f}")
print(f"RMSE : {baseline_rmse:.4f}")
print(f"R²   : {baseline_r2:.4f}")

Training baseline Random Forest...

Baseline validation performance:
MAE  : 16.7545
RMSE : 25.8208
R²   : 0.8165


In [8]:
parameter_combinations = [
    {
        "n_estimators": 150,
        "max_depth": None,
        "min_samples_split": 2,
        "min_samples_leaf": 1,
        "max_features": 1.0
    },
    {
        "n_estimators": 200,
        "max_depth": 20,
        "min_samples_split": 2,
        "min_samples_leaf": 1,
        "max_features": 1.0
    },
    {
        "n_estimators": 200,
        "max_depth": 30,
        "min_samples_split": 2,
        "min_samples_leaf": 1,
        "max_features": 1.0
    },
    {
        "n_estimators": 200,
        "max_depth": None,
        "min_samples_split": 5,
        "min_samples_leaf": 2,
        "max_features": 1.0
    },
    {
        "n_estimators": 200,
        "max_depth": None,
        "min_samples_split": 10,
        "min_samples_leaf": 2,
        "max_features": 0.8
    }
]

print(
    "Number of parameter combinations:",
    len(parameter_combinations)
)

Number of parameter combinations: 5


In [9]:
tuning_results = []

for i, params in enumerate(
    parameter_combinations,
    start=1
):

    print(
        f"\nTraining model {i}/"
        f"{len(parameter_combinations)}"
    )

    print(params)

    model_tuned = RandomForestRegressor(
        random_state=42,
        n_jobs=-1,
        **params
    )

    model_tuned.fit(
        X_tune_train,
        y_tune_train
    )

    validation_predictions = (
        model_tuned.predict(X_validation)
    )

    mae_value = mean_absolute_error(
        y_validation,
        validation_predictions
    )

    rmse_value = np.sqrt(
        mean_squared_error(
            y_validation,
            validation_predictions
        )
    )

    r2_value = r2_score(
        y_validation,
        validation_predictions
    )

    tuning_results.append({
        "Model": i,
        **params,
        "MAE": mae_value,
        "RMSE": rmse_value,
        "R²": r2_value
    })

    print(
        f"MAE={mae_value:.4f}, "
        f"RMSE={rmse_value:.4f}, "
        f"R²={r2_value:.4f}"
    )

tuning_results_df = pd.DataFrame(
    tuning_results
)

print("\nTuning completed.")


Training model 1/5
{'n_estimators': 150, 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 1.0}
MAE=16.6908, RMSE=25.5289, R²=0.8207

Training model 2/5
{'n_estimators': 200, 'max_depth': 20, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 1.0}
MAE=16.6624, RMSE=25.6041, R²=0.8196

Training model 3/5
{'n_estimators': 200, 'max_depth': 30, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 1.0}
MAE=16.6848, RMSE=25.5855, R²=0.8199

Training model 4/5
{'n_estimators': 200, 'max_depth': None, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': 1.0}
MAE=16.5212, RMSE=25.1633, R²=0.8258

Training model 5/5
{'n_estimators': 200, 'max_depth': None, 'min_samples_split': 10, 'min_samples_leaf': 2, 'max_features': 0.8}
MAE=16.1550, RMSE=23.9474, R²=0.8422

Tuning completed.


In [10]:
tuning_results_df.sort_values(
    "R²",
    ascending=False
).reset_index(drop=True)

,Model,n_estimators,max_depth,min_samples_split,min_samples_leaf,max_features,MAE,RMSE,R²
0,5,200,NaN,10,2,0.8,16.155046,23.947384,0.842199
1,4,200,NaN,5,2,1.0,16.521168,25.163345,0.825767
2,1,150,NaN,2,1,1.0,16.690779,25.528935,0.820667
3,3,200,30.0,2,1,1.0,16.684846,25.585523,0.819871
4,2,200,20.0,2,1,1.0,16.662423,25.604060,0.819610


In [11]:
best_result = tuning_results_df.sort_values(
    "R²",
    ascending=False
).iloc[0]

print("Best tuned configuration:")
print(best_result)

Best tuned configuration:
Model                  5.000000
n_estimators         200.000000
max_depth                   NaN
min_samples_split     10.000000
min_samples_leaf       2.000000
max_features           0.800000
MAE                   16.155046
RMSE                  23.947384
R²                     0.842199
Name: 4, dtype: float64


In [12]:
best_params = {
    "n_estimators": int(
        best_result["n_estimators"]
    ),
    "max_depth": (
        None
        if pd.isna(best_result["max_depth"])
        else int(best_result["max_depth"])
    ),
    "min_samples_split": int(
        best_result["min_samples_split"]
    ),
    "min_samples_leaf": int(
        best_result["min_samples_leaf"]
    ),
    "max_features": best_result["max_features"]
}

print("Best parameters:")
print(best_params)

best_model = RandomForestRegressor(
    random_state=42,
    n_jobs=-1,
    **best_params
)

print("\nTraining best tuned model...")

best_model.fit(
    X_tune_train,
    y_tune_train
)

print("Best tuned model trained successfully.")

Best parameters:
{'n_estimators': 200, 'max_depth': None, 'min_samples_split': 10, 'min_samples_leaf': 2, 'max_features': np.float64(0.8)}

Training best tuned model...
Best tuned model trained successfully.


In [13]:
best_validation_predictions = (
    best_model.predict(X_validation)
)

tuned_mae = mean_absolute_error(
    y_validation,
    best_validation_predictions
)

tuned_rmse = np.sqrt(
    mean_squared_error(
        y_validation,
        best_validation_predictions
    )
)

tuned_r2 = r2_score(
    y_validation,
    best_validation_predictions
)

print("Best Tuned Random Forest")
print("=" * 35)

print(f"MAE  : {tuned_mae:.4f} mg/dL")
print(f"RMSE : {tuned_rmse:.4f} mg/dL")
print(f"R²   : {tuned_r2:.4f}")

Best Tuned Random Forest
MAE  : 16.1550 mg/dL
RMSE : 23.9474 mg/dL
R²   : 0.8422


In [14]:
comparison = pd.DataFrame({
    "Model": [
        "Baseline Random Forest",
        "Tuned Random Forest"
    ],
    "MAE": [
        baseline_mae,
        tuned_mae
    ],
    "RMSE": [
        baseline_rmse,
        tuned_rmse
    ],
    "R²": [
        baseline_r2,
        tuned_r2
    ]
})

comparison

,Model,MAE,RMSE,R²
0,Baseline Random Forest,16.754476,25.820842,0.816543
1,Tuned Random Forest,16.155046,23.947384,0.842199


In [15]:
mae_improvement = (
    (baseline_mae - tuned_mae)
    / baseline_mae
) * 100

rmse_improvement = (
    (baseline_rmse - tuned_rmse)
    / baseline_rmse
) * 100

r2_improvement = (
    (tuned_r2 - baseline_r2)
    / abs(baseline_r2)
) * 100

print(
    f"MAE improvement : {mae_improvement:.2f}%"
)

print(
    f"RMSE improvement: {rmse_improvement:.2f}%"
)

print(
    f"R² improvement  : {r2_improvement:.2f}%"
)

MAE improvement : 3.58%
RMSE improvement: 7.26%
R² improvement  : 3.14%


In [16]:
final_predictions = best_model.predict(
    X_test
)

final_mae = mean_absolute_error(
    y_test,
    final_predictions
)

final_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        final_predictions
    )
)

final_r2 = r2_score(
    y_test,
    final_predictions
)

print("FINAL TEST PERFORMANCE")
print("=" * 35)

print(f"MAE  : {final_mae:.4f} mg/dL")
print(f"RMSE : {final_rmse:.4f} mg/dL")
print(f"R²   : {final_r2:.4f}")

FINAL TEST PERFORMANCE
MAE  : 17.2889 mg/dL
RMSE : 25.6017 mg/dL
R²   : 0.8266


In [17]:
final_comparison = pd.DataFrame({
    "Model": [
        "Baseline Random Forest",
        "Tuned Random Forest"
    ],
    "MAE": [
        17.5745,
        final_mae
    ],
    "RMSE": [
        26.4067,
        final_rmse
    ],
    "R²": [
        0.8155,
        final_r2
    ]
})

final_comparison

,Model,MAE,RMSE,R²
0,Baseline Random Forest,17.574500,26.40670,0.81550
1,Tuned Random Forest,17.288891,25.60171,0.82662


In [18]:
if final_r2 > 0.8155 and final_mae < 17.5745:

    print("🎉 Tuned model improved the baseline.")

elif final_r2 > 0.8155:

    print(
        "Tuned model improved R² "
        "but not MAE."
    )

elif final_mae < 17.5745:

    print(
        "Tuned model improved MAE "
        "but not R²."
    )

else:

    print(
        "Baseline Random Forest remains "
        "the stronger model."
    )

🎉 Tuned model improved the baseline.


In [19]:
MODEL_DIR = "../models"

os.makedirs(
    MODEL_DIR,
    exist_ok=True
)

if (
    final_r2 > 0.8155
    and final_mae < 17.5745
):

    final_model_path = (
        "../models/"
        "improved_random_forest_glucose_model.pkl"
    )

    joblib.dump(
        best_model,
        final_model_path
    )

    joblib.dump(
        feature_columns,
        "../models/feature_columns.pkl"
    )

    print(
        "Improved model saved to:"
    )

    print(final_model_path)

else:

    print(
        "Baseline model remains the "
        "best model. No replacement made."
    )

Improved model saved to:
../models/improved_random_forest_glucose_model.pkl


# Model Improvement Summary

## Baseline Random Forest

- MAE: 17.57 mg/dL
- RMSE: 26.41 mg/dL
- R²: 0.8155

## Tuning

Random Forest hyperparameters were systematically evaluated using a chronological validation set.

The final test set remained untouched during hyperparameter selection.

## Final Decision

The tuned Random Forest is selected only if it demonstrates improvement over the baseline on the final test set.

## Evaluation Principle

The test set is used only once for final model comparison to reduce the risk of overfitting and optimistic performance estimates.

## Next Step

If Random Forest tuning provides limited improvement, advanced gradient boosting models can be evaluated against the established baseline.